### Dataset Corrections

In [114]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/02_interim/dataset_without_leakage.csv', low_memory=False)
df

,statusId,status,status_group,constructorId,constructorRef,name_constructor,nationality_constructor,url_constructor,driverId,driverRef,...,fp3_time,quali_date,quali_time,sprint_date,sprint_time,resultId,number,grid,positionOrder,laps
0,1,Finished,Finished,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,hamilton,...,\N,\N,\N,\N,\N,7580,1,12,7,31
1,1,Finished,Finished,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,hamilton,...,\N,\N,\N,\N,\N,7599,1,9,6,56
2,1,Finished,Finished,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,hamilton,...,\N,\N,\N,\N,\N,7617,1,5,4,57
3,1,Finished,Finished,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,hamilton,...,\N,\N,\N,\N,\N,7686,1,16,13,58
4,1,Finished,Finished,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,1,hamilton,...,\N,\N,\N,\N,\N,7734,1,4,1,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,140,Undertray,Others,3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...,849,latifi,...,\N,2022-07-08,15:00:00,2022-07-09,14:30:00,25624,6,17,19,48
26755,140,Undertray,Others,6,ferrari,Ferrari,Italian,http://en.wikipedia.org/wiki/Scuderia_Ferrari,844,leclerc,...,09:30:00,2023-08-26,13:00:00,\N,\N,26104,16,9,19,41
26756,140,Undertray,Others,117,aston_martin,Aston Martin,British,http://en.wikipedia.org/wiki/Aston_Martin_in_F...,4,alonso,...,\N,2023-10-20,21:00:00,2023-10-21,22:00:00,26201,14,0,16,49
26757,140,Undertray,Others,213,alphatauri,AlphaTauri,Italian,http://en.wikipedia.org/wiki/Scuderia_AlphaTauri,852,tsunoda,...,11:00:00,2022-07-23,14:00:00,\N,\N,25645,22,8,20,17


#### 1. Grid Position

The dataset saves the exit from pit-lane as zero, what will cause the ML model to learn: "Exit from pit-lane is better than exit from grid position 1". Because of this, the grid position 0 will be changed to max grid pos + 1, based on each race individually, since grid size varies from 1950 to nowadays.

In [123]:
df['grid'] = df['grid'].replace(0, np.nan)

max_grid_per_race = df.groupby('raceId')['grid'].transform('max') + 1

df['grid'] = df['grid'].fillna(max_grid_per_race)
df['grid'] = df['grid'].astype(int)

#### 2. Drop of irrelevant columns for learning
Now, columns without relevant info to the algorithm learn will be dropped. `constructor_url`, for example.

In [116]:
# Constructor Info
df = df.drop(['constructorRef', 'name_constructor', 'nationality_constructor', 'url_constructor'], axis=1)

# Driver Info
df = df.drop(['driverRef', 'number_driver', 'code', 'forename', 'surname', 'dob', 'nationality', 'url_driver'], axis=1)

# Race Info
df = df.drop(['name', 'time_race', 'url'], axis=1)

# FP info
df = df.drop(['fp1_time', 'fp2_time', 'fp3_time'], axis=1)

# Quali and Sprint info
df = df.drop(['quali_time', 'sprint_time'], axis=1)

# Results info
df = df.drop(['number'], axis=1)

df

,statusId,status,status_group,constructorId,driverId,raceId,year,round,circuitId,date,fp1_date,fp2_date,fp3_date,quali_date,sprint_date,resultId,grid,positionOrder,laps
0,1,Finished,Finished,1,1,2,2009,2,2,2009-04-05,\N,\N,\N,\N,\N,7580,12.0,7,31
1,1,Finished,Finished,1,1,3,2009,3,17,2009-04-19,\N,\N,\N,\N,\N,7599,9.0,6,56
2,1,Finished,Finished,1,1,4,2009,4,3,2009-04-26,\N,\N,\N,\N,\N,7617,5.0,4,57
3,1,Finished,Finished,1,1,7,2009,7,5,2009-06-07,\N,\N,\N,\N,\N,7686,16.0,13,58
4,1,Finished,Finished,1,1,10,2009,10,11,2009-07-26,\N,\N,\N,\N,\N,7734,4.0,1,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,140,Undertray,Others,3,849,1084,2022,11,70,2022-07-10,2022-07-08,2022-07-09,\N,2022-07-08,2022-07-09,25624,17.0,19,48
26755,140,Undertray,Others,6,844,1111,2023,13,39,2023-08-27,2023-08-25,2023-08-25,2023-08-26,2023-08-26,\N,26104,9.0,19,41
26756,140,Undertray,Others,117,4,1116,2023,18,69,2023-10-22,2023-10-20,2023-10-21,\N,2023-10-20,2023-10-21,26201,17.0,16,49
26757,140,Undertray,Others,213,852,1085,2022,12,34,2022-07-24,2022-07-22,2022-07-22,2022-07-23,2022-07-23,\N,25645,8.0,20,17


#### 3. Data Typing

In [117]:
df = df.replace(to_replace='\\N', value=np.nan)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   statusId       26759 non-null  int64  
 1   status         26759 non-null  str    
 2   status_group   26759 non-null  str    
 3   constructorId  26759 non-null  int64  
 4   driverId       26759 non-null  int64  
 5   raceId         26759 non-null  int64  
 6   year           26759 non-null  int64  
 7   round          26759 non-null  int64  
 8   circuitId      26759 non-null  int64  
 9   date           26759 non-null  str    
 10  fp1_date       1799 non-null   str    
 11  fp2_date       1799 non-null   str    
 12  fp3_date       1439 non-null   str    
 13  quali_date     1799 non-null   str    
 14  sprint_date    360 non-null    str    
 15  resultId       26759 non-null  int64  
 16  grid           26759 non-null  float64
 17  positionOrder  26759 non-null  int64  
 18  laps           26

In [118]:
df

,statusId,status,status_group,constructorId,driverId,raceId,year,round,circuitId,date,fp1_date,fp2_date,fp3_date,quali_date,sprint_date,resultId,grid,positionOrder,laps
0,1,Finished,Finished,1,1,2,2009,2,2,2009-04-05,NaN,NaN,NaN,NaN,NaN,7580,12.0,7,31
1,1,Finished,Finished,1,1,3,2009,3,17,2009-04-19,NaN,NaN,NaN,NaN,NaN,7599,9.0,6,56
2,1,Finished,Finished,1,1,4,2009,4,3,2009-04-26,NaN,NaN,NaN,NaN,NaN,7617,5.0,4,57
3,1,Finished,Finished,1,1,7,2009,7,5,2009-06-07,NaN,NaN,NaN,NaN,NaN,7686,16.0,13,58
4,1,Finished,Finished,1,1,10,2009,10,11,2009-07-26,NaN,NaN,NaN,NaN,NaN,7734,4.0,1,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,140,Undertray,Others,3,849,1084,2022,11,70,2022-07-10,2022-07-08,2022-07-09,NaN,2022-07-08,2022-07-09,25624,17.0,19,48
26755,140,Undertray,Others,6,844,1111,2023,13,39,2023-08-27,2023-08-25,2023-08-25,2023-08-26,2023-08-26,NaN,26104,9.0,19,41
26756,140,Undertray,Others,117,4,1116,2023,18,69,2023-10-22,2023-10-20,2023-10-21,NaN,2023-10-20,2023-10-21,26201,17.0,16,49
26757,140,Undertray,Others,213,852,1085,2022,12,34,2022-07-24,2022-07-22,2022-07-22,2022-07-23,2022-07-23,NaN,25645,8.0,20,17


Fixing Column Types

In [124]:
df['date'] = pd.to_datetime(df['date'])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26759 entries, 0 to 26758
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   statusId       26759 non-null  int64         
 1   status         26759 non-null  str           
 2   status_group   26759 non-null  str           
 3   constructorId  26759 non-null  int64         
 4   driverId       26759 non-null  int64         
 5   raceId         26759 non-null  int64         
 6   year           26759 non-null  int64         
 7   round          26759 non-null  int64         
 8   circuitId      26759 non-null  int64         
 9   date           26759 non-null  datetime64[us]
 10  fp1_date       1799 non-null   str           
 11  fp2_date       1799 non-null   str           
 12  fp3_date       1439 non-null   str           
 13  quali_date     1799 non-null   str           
 14  sprint_date    360 non-null    str           
 15  resultId       26759 non-null 

#### Saving updated dataset

In [125]:
df.to_csv('../data/02_interim/dataset_after_corrections.csv', index=False)